<a href="https://colab.research.google.com/github/AzadMahmud/dl-lab/blob/main/assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers


In [2]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "code":
        ROOT = ROOT.parent

OUT = ROOT / "outputs"
M_DIR, C_DIR, MO_DIR = OUT / "metrics", OUT / "curves", OUT / "models"
for d in [M_DIR, C_DIR, MO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [3]:
def data(subset: int | None = None):
    (xtr, ytr), (xte, yte) = tf.keras.datasets.cifar10.load_data()
    xtr, xte = xtr.astype("float32") / 255.0, xte.astype("float32") / 255.0
    ytr, yte = tf.keras.utils.to_categorical(ytr, 10), tf.keras.utils.to_categorical(yte, 10)
    xval, yval = xtr[-5000:], ytr[-5000:]
    xtr, ytr = xtr[:-5000], ytr[:-5000]
    if subset:
        xtr, ytr = xtr[:subset], ytr[:subset]
    return (xtr, ytr), (xval, yval), (xte, yte)

In [4]:
def make_model(kind: str):
    l2 = 1e-4 if kind in {"reg_only", "reg_dropout"} else 0.0
    dr = 0.4 if kind in {"dropout_only", "reg_dropout"} else 0.0
    reg = regularizers.l2(l2) if l2 else None

    if kind == "underfit":
        m = models.Sequential(
            [
                layers.Input((32, 32, 3)),
                layers.Conv2D(8, 3, activation="relu"),
                layers.Flatten(),
                layers.Dense(10, activation="softmax"),
            ]
        )
    else:
        m = models.Sequential(
            [
                layers.Input((32, 32, 3)),
                layers.Conv2D(32, 3, padding="same", activation="relu", kernel_regularizer=reg),
                layers.Conv2D(32, 3, padding="same", activation="relu", kernel_regularizer=reg),
                layers.MaxPooling2D(),
                layers.Dropout(dr),
                layers.Conv2D(64, 3, padding="same", activation="relu", kernel_regularizer=reg),
                layers.Conv2D(64, 3, padding="same", activation="relu", kernel_regularizer=reg),
                layers.MaxPooling2D(),
                layers.Dropout(dr),
                layers.Conv2D(128, 3, padding="same", activation="relu", kernel_regularizer=reg),
                layers.Conv2D(128, 3, padding="same", activation="relu", kernel_regularizer=reg),
                layers.GlobalAveragePooling2D(),
                layers.Dense(128, activation="relu", kernel_regularizer=reg),
                layers.Dropout(dr),
                layers.Dense(10, activation="softmax"),
            ]
        )

    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    return m


In [5]:
def train_one(kind: str):
    subset = 3000 if kind == "underfit" else None
    epochs = 4 if kind == "underfit" else 20
    bs = 128
    (xtr, ytr), (xval, yval), (xte, yte) = data(subset=subset)
    m = make_model(kind)

    cbs = []
    if kind != "overfit":
        cbs = [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=3, restore_best_weights=True)]

    t0 = time.time()
    h = m.fit(xtr, ytr, validation_data=(xval, yval), epochs=epochs, batch_size=bs, verbose=1, callbacks=cbs)
    t1 = time.time()
    te_loss, te_acc = m.evaluate(xte, yte, verbose=0)

    hist = pd.DataFrame(h.history)
    hist.to_csv(M_DIR / f"history_{kind}.csv", index=False)
    m.save(MO_DIR / f"{kind}.keras")

    res = {
        "scenario": kind,
        "epochs_ran": int(len(hist)),
        "train_acc_last": float(hist["accuracy"].iloc[-1]),
        "val_acc_last": float(hist["val_accuracy"].iloc[-1]),
        "best_val_acc": float(hist["val_accuracy"].max()),
        "gap_train_minus_val": float(hist["accuracy"].iloc[-1] - hist["val_accuracy"].iloc[-1]),
        "test_acc": float(te_acc),
        "test_loss": float(te_loss),
        "train_time_sec": float(t1 - t0),
    }
    tf.keras.backend.clear_session()
    return res


In [6]:
def plot_curves(all_hist: dict[str, pd.DataFrame]):
    plt.figure(figsize=(10, 4))
    for k, h in all_hist.items():
        plt.plot(h["val_accuracy"], label=k)
    plt.title("Validation Accuracy Comparison")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(C_DIR / "val_accuracy_all.png", dpi=220)
    plt.close()

    plt.figure(figsize=(10, 4))
    for k, h in all_hist.items():
        plt.plot(h["val_loss"], label=k)
    plt.title("Validation Loss Comparison")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(C_DIR / "val_loss_all.png", dpi=220)
    plt.close()

In [7]:
def main():
    scenarios = ["underfit", "overfit", "reg_only", "dropout_only", "reg_dropout"]
    rows, all_hist = [], {}
    for s in scenarios:
        print("Running", s)
        rows.append(train_one(s))
        all_hist[s] = pd.read_csv(M_DIR / f"history_{s}.csv")

    df = pd.DataFrame(rows).sort_values("test_acc", ascending=False)
    df.to_csv(M_DIR / "scenario_comparison.csv", index=False)
    plot_curves(all_hist)

    best = df.iloc[0]
    (M_DIR / "summary.json").write_text(
        json.dumps(
            {
                "best_scenario": best["scenario"],
                "best_test_acc": float(best["test_acc"]),
                "note": "Underfit should show low train+val accuracy; overfit should show larger train-val gap.",
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Done. Outputs saved in", OUT)


if __name__ == "__main__":
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    main()

Running underfit
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/4
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 89ms/step - accuracy: 0.2097 - loss: 2.1799 - val_accuracy: 0.2904 - val_loss: 2.0093
Epoch 2/4
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3693 - loss: 1.8602 - val_accuracy: 0.3556 - val_loss: 1.8565
Epoch 3/4
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4313 - loss: 1.7072 - val_accuracy: 0.3688 - val_loss: 1.8143
Epoch 4/4
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4693 - loss: 1.6030 - val_accuracy: 0.3850 - val_loss: 1.7819
Running overfit
Epoch 1/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.3010 - loss: 1.8394 - val_accuracy: 0.4096 - val_loss: 1.5564
Epoch 2/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.4836 - loss: 1.3862 - val_accuracy: 0.5522 - val_loss: 1.2157
Epoch 3/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.5721 - loss: 1.1704 - val_accuracy: 0.6058 - val_loss: 1.0831
Epoch 4/20
352/352 ━━